In [1]:
import os

LANGS = {
    "eng": "eng_Latn",
    "hin": "hin_Deva",
    "tam": "tam_Taml",
    "tel": "tel_Telu",
}
SPLIT = "dev"

# path to the already-extracted flores folder, one level up from partA/
flores_path = "../flores200_dataset"
outdir = "corpus_real"
os.makedirs(outdir, exist_ok=True)

line_counts = {}
for short, code in LANGS.items():
    src_path = f"{flores_path}/{SPLIT}/{code}.{SPLIT}"
    with open(src_path, encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]
    with open(f"{outdir}/{short}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    line_counts[short] = len(lines)
    print(f"{short}: wrote {len(lines)} lines")

print(f"\nline counts match: {len(set(line_counts.values())) == 1}")

eng: wrote 997 lines
hin: wrote 997 lines
tam: wrote 997 lines
tel: wrote 997 lines

line counts match: True


In [2]:
!pip install tiktoken regex -q

In [3]:
import unicodedata
import tiktoken

try:
    import regex
    HAVE_REGEX = True
except ImportError:
    HAVE_REGEX = False


def read_lines(path):
    lines = []
    with open(path, encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if line:
                lines.append(unicodedata.normalize("NFC", line))
    return lines


def count_words(line, ws_split):
    return len(line.split()) if ws_split else len(line.split(" "))


def count_chars(line, grapheme):
    if grapheme and HAVE_REGEX:
        return len(regex.findall(r"\X", line))
    return len(line)


def analyze(lines, encode, no_lowercase=False, ws_split=False,
            grapheme_chars=False, micro_average=False, denominator="word"):
    total_tokens = total_words = total_graphs = total_bytes = total_sents = 0
    per_line = []

    for line in lines:
        if not no_lowercase:
            line = line.lower()
        tokens = encode(line)
        n_tok = len(tokens)
        n_words = count_words(line, ws_split)
        n_graphs = count_chars(line, grapheme_chars)
        n_bytes = len(line.encode("utf-8"))

        total_tokens += n_tok
        total_words += n_words
        total_graphs += n_graphs
        total_bytes += n_bytes
        total_sents += 1

        denom = {"word": n_words, "grapheme": n_graphs, "byte": n_bytes, "sentence": 1}[denominator]
        if denom > 0:
            per_line.append(n_tok / denom)

    if micro_average:
        denom_total = {"word": total_words, "grapheme": total_graphs,
                        "byte": total_bytes, "sentence": total_sents}[denominator]
        fertility = total_tokens / denom_total
    else:
        fertility = sum(per_line) / len(per_line)

    return fertility


def run(corpus_paths, tokenizer_encode, **flags):
    results = {}
    for lang, path in corpus_paths.items():
        lines = read_lines(path)
        results[lang] = analyze(lines, tokenizer_encode, **flags)
    return results


CORPUS = {
    "eng": "corpus_real/eng.txt",
    "hin": "corpus_real/hin.txt",
    "tam": "corpus_real/tam.txt",
    "tel": "corpus_real/tel.txt",
}

gpt2_enc = tiktoken.get_encoding("gpt2").encode
print("functions loaded, ready to run")

functions loaded, ready to run


In [4]:
baseline = run(CORPUS, gpt2_enc)
print("BASELINE (original buggy settings):")
for lang, fert in baseline.items():
    print(f"  {lang}: {fert:.4f}")

BASELINE (original buggy settings):
  eng: 1.2825
  hin: 7.8232
  tam: 24.7332
  tel: 20.3995


A2 evidence generation


In [5]:
ablations = {
    "no-lowercase":  dict(no_lowercase=True),
    "ws-split":      dict(ws_split=True),
    "micro-average": dict(micro_average=True),
    "ALL FIXES":     dict(no_lowercase=True, ws_split=True, grapheme_chars=True, micro_average=True),
}

print(f"{'ablation':<18}" + "".join(f"{l:>10}" for l in CORPUS))
print("-" * (18 + 10*len(CORPUS)))
print(f"{'baseline':<18}" + "".join(f"{baseline[l]:>10.4f}" for l in CORPUS))

for name, flags in ablations.items():
    result = run(CORPUS, gpt2_enc, **flags)
    print(f"{name:<18}" + "".join(f"{result[l]:>10.4f}" for l in CORPUS))

ablation                 eng       hin       tam       tel
----------------------------------------------------------
baseline              1.2825    7.8232   24.7332   20.3995
no-lowercase          1.2367    7.8225   24.7314   20.3936
ws-split              1.2826    7.8260   24.8669   20.6243
micro-average         1.2740    7.7934   24.4650   20.2276
ALL FIXES             1.2285    7.7957   24.6165   20.4810


print(f"{'lang':<6}{'len()_denominator':>20}{'grapheme_denominator':>22}{'% difference':>14}")
for lang, path in CORPUS.items():
    lines = read_lines(path)
    f_codepoint = analyze(lines, gpt2_enc, denominator="grapheme", grapheme_chars=False)
    f_grapheme  = analyze(lines, gpt2_enc, denominator="grapheme", grapheme_chars=True)
    pct = (f_grapheme - f_codepoint) / f_codepoint * 100
    print(f"{lang:<6}{f_codepoint:>20.4f}{f_grapheme:>22.4f}{pct:>13.2f}%")

A3 step

In [8]:
!pip install transformers sentencepiece -q

In [9]:
from transformers import AutoTokenizer

muril_tok = AutoTokenizer.from_pretrained("google/muril-base-cased")
muril_encode = lambda s: muril_tok.encode(s, add_special_tokens=False)
print("muril loaded")

muril loaded


In [10]:
tokenizers = {
    "gpt2": gpt2_enc,
    "muril": muril_encode,
}

denominators = ["word", "grapheme", "byte", "sentence"]

print(f"{'tokenizer':<10}{'denom':<10}" + "".join(f"{l:>10}" for l in CORPUS))
print("-" * (20 + 10*len(CORPUS)))

results = {}
for tok_name, enc in tokenizers.items():
    for denom in denominators:
        row = {}
        for lang, path in CORPUS.items():
            lines = read_lines(path)
            fert = analyze(lines, enc, no_lowercase=True, ws_split=True,
                            grapheme_chars=True, micro_average=True,
                            denominator=denom)
            row[lang] = fert
        results[(tok_name, denom)] = row
        print(f"{tok_name:<10}{denom:<10}" + "".join(f"{row[l]:>10.4f}" for l in CORPUS))

tokenizer denom            eng       hin       tam       tel
------------------------------------------------------------
gpt2      word          1.2285    7.7957   24.6165   20.4810
gpt2      grapheme      0.2056    2.3279    4.2043    4.5623
gpt2      byte          0.2055    0.5946    0.9959    0.9907
gpt2      sentence     25.8185  192.4052  398.3581  336.6520
muril     word          1.2582    1.2455    1.7225    1.9547
muril     grapheme      0.2106    0.3719    0.2942    0.4354
muril     byte          0.2104    0.0950    0.0697    0.0946
muril     sentence     26.4443   30.7412   27.8746   32.1304


In [11]:
print(f"\n{'tokenizer':<10}{'denom':<10}{'hin/eng':>10}{'tam/eng':>10}{'tel/eng':>10}")
for (tok_name, denom), row in results.items():
    print(f"{tok_name:<10}{denom:<10}{row['hin']/row['eng']:>10.2f}"
          f"{row['tam']/row['eng']:>10.2f}{row['tel']/row['eng']:>10.2f}")


tokenizer denom        hin/eng   tam/eng   tel/eng
gpt2      word            6.35     20.04     16.67
gpt2      grapheme       11.32     20.45     22.19
gpt2      byte            2.89      4.85      4.82
gpt2      sentence        7.45     15.43     13.04
muril     word            0.99      1.37      1.55
muril     grapheme        1.77      1.40      2.07
muril     byte            0.45      0.33      0.45
muril     sentence        1.16      1.05      1.22
